# Prim and Dijkstra - one skeleton, two algorithms

Both algorithms grow a tree outward from a source, and at every step they settle the
cheapest vertex that is not settled yet. They differ in **one line**: what "cheapest"
means.

| | priority of a newly seen vertex `v` | answers |
|---|---|---|
| **Prim** | the weight of the edge `u -> v` | which edges connect everything most cheaply |
| **Dijkstra** | `dist[u] + weight(u -> v)` | how far every vertex is from the source |

Same heap, same settle loop, same `O(E log V)`. Day 15 solved the first problem with
Kruskal instead; the graph below is the same one, so the two answers can be compared.

In [1]:
from heapq import heappush, heappop
INF = float('inf')

NAMES = 'ABCDEFG'
EDGES = [(0, 1, 7), (0, 3, 5), (1, 2, 8), (1, 3, 9), (1, 4, 7),
         (2, 4, 5), (3, 4, 15), (3, 5, 6), (4, 5, 8), (4, 6, 9), (5, 6, 11)]

def build_adj(n, edges, directed=False):
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w))
        if not directed:
            adj[v].append((u, w))
    return adj

adj = build_adj(len(NAMES), EDGES)
for u, row in enumerate(adj):
    print(NAMES[u], '->', ' '.join('%s(%d)' % (NAMES[v], w) for v, w in row))

A -> B(7) D(5)
B -> A(7) C(8) D(9) E(7)
C -> B(8) E(5)
D -> A(5) B(9) E(15) F(6)
E -> B(7) C(5) D(15) F(8) G(9)
F -> D(6) E(8) G(11)
G -> E(9) F(11)


## The shared skeleton

`key(d, w)` is the only parameter that matters. `best[v]` holds the current best
priority for `v`, `parent[v]` the vertex it was reached from, and `done[v]` marks a
vertex as **settled** - its value can never improve again.

The heap is used with **lazy deletion**: instead of decreasing a key in place (which
`heapq` cannot do), a better entry is simply pushed and the stale one is skipped when
it surfaces. That is why the loop starts with `if done[u]: continue`.

In [2]:
def heap_greedy(n, adj, src, key):
    best, parent, done, order = [INF] * n, [-1] * n, [False] * n, []
    best[src] = 0
    pq = [(0, src)]
    while pq:
        d, u = heappop(pq)
        if done[u]:                     # stale entry from an earlier improvement
            continue
        done[u] = True
        order.append(u)
        for v, w in adj[u]:
            nd = key(d, w)              # <- Prim: w      Dijkstra: d + w
            if not done[v] and nd < best[v]:
                best[v], parent[v] = nd, u
                heappush(pq, (nd, v))
    return best, parent, order

prim_key     = lambda d, w: w          # the edge on its own
dijkstra_key = lambda d, w: d + w      # the whole path so far

for name, key in [('Prim', prim_key), ('Dijkstra', dijkstra_key)]:
    # reaching a vertex that sits 5 away, over an edge of weight 3:
    print('%-9s priority = %d' % (name, key(5, 3)))

Prim      priority = 3
Dijkstra  priority = 8


## Prim: the same MST day 15 found with Kruskal

In [3]:
def prim(n, adj, src=0):
    best, parent, order = heap_greedy(n, adj, src, lambda d, w: w)
    tree = sorted((best[v], parent[v], v) for v in range(n) if parent[v] != -1)
    return tree, sum(w for w, _, _ in tree), order

tree, total, order = prim(len(NAMES), adj)
print('settle order:', ' '.join(NAMES[u] for u in order))
for w, p, v in tree:
    print('   %s%s (%d)' % (NAMES[p], NAMES[v], w))
print('total       :', total)
print('day 15 Kruskal on this graph: 39 - same weight, found in a different order')

settle order: A D F B E C G
   AD (5)
   EC (5)
   DF (6)
   AB (7)
   BE (7)
   EG (9)
total       : 39
day 15 Kruskal on this graph: 39 - same weight, found in a different order


## Dijkstra: change one lambda

Nothing else moves. The values coming out are now distances from `A`, and the parents
form a **shortest-path tree**.

In [4]:
def dijkstra(n, adj, src=0):
    return heap_greedy(n, adj, src, lambda d, w: d + w)

def path_to(parent, v):
    out = []
    while v != -1:
        out.append(v); v = parent[v]
    return ' -> '.join(NAMES[x] for x in out[::-1])

dist, parent, order = dijkstra(len(NAMES), adj)
print('settle order:', ' '.join(NAMES[u] for u in order))
for v in range(len(NAMES)):
    print('   %s : %-3d via %s' % (NAMES[v], dist[v], path_to(parent, v)))

settle order: A D B F E C G
   A : 0   via A
   B : 7   via A -> B
   C : 15  via A -> B -> C
   D : 5   via A -> D
   E : 14  via A -> B -> E
   F : 11  via A -> D -> F
   G : 22  via A -> D -> F -> G


## The two trees are not the same tree

Both are spanning trees of the same graph, but they optimise different things: the MST
minimises the **total** weight, the shortest-path tree minimises **each distance from
the source**. Walking the MST from A to G costs more than the true shortest path, and
the shortest-path tree weighs more in total.

In [5]:
mst_edges = {tuple(sorted((p, v))) for _, p, v in tree}
sp_edges  = {tuple(sorted((parent[v], v))) for v in range(len(NAMES)) if parent[v] != -1}
fmt = lambda s: ' '.join(NAMES[a] + NAMES[b] for a, b in sorted(s))
print('Prim     :', fmt(mst_edges))
print('Dijkstra :', fmt(sp_edges))
print('only in the shortest-path tree :', fmt(sp_edges - mst_edges))
print('only in the MST                :', fmt(mst_edges - sp_edges))

def tree_path_cost(edges, u, v):
    g = {}
    for a, b, w in edges:
        g.setdefault(a, []).append((b, w)); g.setdefault(b, []).append((a, w))
    stack, seen = [(u, 0)], {u}
    while stack:
        x, acc = stack.pop()
        if x == v:
            return acc
        for y, w in g.get(x, []):
            if y not in seen:
                seen.add(y); stack.append((y, acc + w))
    return INF

mst_t = [(p, v, w) for w, p, v in tree]
print('A->G along the MST :', tree_path_cost(mst_t, 0, 6), ' shortest path :', dist[6])
sp_total = sum(w for u, v, w in EDGES if tuple(sorted((u, v))) in sp_edges)
print('total weight: MST %d, shortest-path tree %d' % (total, sp_total))

Prim     : AB AD BE CE DF EG
Dijkstra : AB AD BC BE DF FG
only in the shortest-path tree : BC FG
only in the MST                : CE EG
A->G along the MST : 23  shortest path : 22
total weight: MST 39, shortest-path tree 44


## Where Dijkstra breaks: a negative edge

Dijkstra's correctness rests on one assumption: once a vertex is popped with the
smallest tentative distance, no other route can beat it - because every remaining route
starts from something already at least that far away and can only get longer. A
negative edge destroys that argument.

Prim is unaffected: it never adds distances together, so an edge weight of -10 is just
a very attractive edge.

In [6]:
def bellman_ford(n, edges, src, directed=True):
    dist = [INF] * n
    dist[src] = 0
    arcs = list(edges) + ([] if directed else [(v, u, w) for u, v, w in edges])
    for _ in range(n - 1):
        for u, v, w in arcs:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist

neg = [(0, 1, 5), (0, 2, 2), (1, 2, -10)]     # S->A 5, S->B 2, A->B -10
dneg, _, _ = dijkstra(3, build_adj(3, neg, directed=True), 0)
print('Dijkstra     :', dneg, '<- wrong')
print('Bellman-Ford :', bellman_ford(3, neg, 0), '<- right')
print('B is settled at 2; when A is settled later, the -10 edge finds B already done.')

Dijkstra     : [0, 5, 2] <- wrong
Bellman-Ford : [0, 5, -5] <- right
B is settled at 2; when A is settled later, the -10 edge finds B already done.


## LeetCode 743 - Network Delay Time

A signal starts at node `k` and travels along directed edges. How long until every node
has it? That is Dijkstra from `k`, then take the maximum distance - and `-1` if any node
is still at infinity.

In [7]:
def network_delay_time(times, n, k):
    adj = build_adj(n, [(u - 1, v - 1, w) for u, v, w in times], directed=True)
    dist, _, _ = dijkstra(n, adj, k - 1)
    slowest = max(dist)
    return -1 if slowest == INF else slowest

for times, n, k in [([[2,1,1],[2,3,1],[3,4,1]], 4, 2), ([[1,2,1]], 2, 1), ([[1,2,1]], 2, 2)]:
    print(times, 'k =', k, '->', network_delay_time(times, n, k))

[[2, 1, 1], [2, 3, 1], [3, 4, 1]] k = 2 -> 2
[[1, 2, 1]] k = 1 -> 1
[[1, 2, 1]] k = 2 -> -1


## Complexity

Every edge can push at most one entry, so the heap holds `O(E)` entries and each pop or
push is `O(log E) = O(log V)`: **`O(E log V)`** time, `O(V + E)` space. On a dense graph
the plain `O(V^2)` array version of either algorithm is faster, because the heap stops
paying for itself once `E` approaches `V^2`.

## Tests

In [8]:
assert total == 39
assert dist == bellman_ford(len(NAMES), EDGES, 0, directed=False)
assert dist[2] == 15 and dist[6] == 22
assert mst_edges != sp_edges and sp_total > total
assert tree_path_cost(mst_t, 0, 6) == 23 > dist[6]
assert dneg[2] == 2 and bellman_ford(3, neg, 0)[2] == -5
assert network_delay_time([[2,1,1],[2,3,1],[3,4,1]], 4, 2) == 2
assert network_delay_time([[1,2,1]], 2, 2) == -1
print('all assertions passed')

all assertions passed
